# 🏛️ DỰ ÁN VILAW-LLM: HUẤN LUYỆN SFT VỚI UNSLOTH & QLoRA
### Base Model: `Qwen/Qwen2.5-7B-Instruct` | Hardware: Google Colab T4 GPU (Free)

> **Pipeline gồm 6 bước:**
> 1. Cài đặt thư viện Unsloth & TRL tối ưu riêng cho GPU T4.
> 2. Nạp Base Model 4-bit (QLoRA) tiết kiệm 70% VRAM.
> 3. Gắn LoRA Adapter (r=16, alpha=32) vào Attention & FFN layers.
> 4. Định dạng dữ liệu pháp lý theo chuẩn ChatML template.
> 5. Huấn luyện với SFTTrainer.
> 6. Kiểm thử suy luận (Inference Test) & Lưu LoRA Adapter.

## 1. Kiểm tra GPU & Cài đặt Unsloth

In [ ]:
# Kiểm tra GPU Tesla T4 16GB
!nvidia-smi

In [ ]:
# Cài đặt Unsloth và TRL bản tương thích tối ưu cho Colab
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" trl peft accelerate bitsandbytes
!pip install pyarrow pandas datasets

## 2. Nạp Base Model với Unsloth 4-bit (QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None        # Tự động nhận diện (Float16 cho T4)
load_in_4bit = True # Bật 4-bit quantization

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

## 3. Cấu hình LoRA Adapter (PEFT)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 4. Chuẩn bị Dữ liệu Huấn luyện & ChatML Template

In [ ]:
import pandas as pd
from datasets import Dataset
from unsloth.chat_templates import get_chat_template

# 1. Gắn ChatML template chuẩn của Qwen2.5 vào tokenizer
tokenizer = get_chat_template(tokenizer, chat_template="chatml")

SYSTEM_PROMPT = (
    "Bạn là một chuyên gia tư vấn pháp luật Việt Nam am hiểu sâu sắc các quy định pháp luật. "
    "Hãy trả lời câu hỏi dựa trên các văn bản quy phạm pháp luật hiện hành, "
    "viện dẫn chính xác số Điều, Khoản, tên luật và đưa ra lập luận logic, rõ ràng."
)

# 2. Đọc dữ liệu từ file Parquet (bạn upload file này lên Colab qua thanh Files bên trái)
df = pd.read_parquet("legal_sft_train.parquet")
print(f"Tổng số mẫu dữ liệu sẵn có: {len(df):,}")

# Lấy 10,000 mẫu đại diện cho đợt train đầu tiên
df_sample = df.sample(n=min(10000, len(df)), random_state=42)

formatted_data = []
for _, row in df_sample.iterrows():
    formatted_data.append({
        "conversations": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]}
        ]
    })

dataset = Dataset.from_pandas(pd.DataFrame(formatted_data))

def apply_template(examples):
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in examples["conversations"]
    ]
    return {"text": texts}

dataset = dataset.map(apply_template, batched=True)
print("Ví dụ 1 mẫu sau khi áp ChatML template:")
print(dataset[0]["text"][:500])

## 5. Khởi tạo SFTTrainer & Bắt đầu Huấn luyện

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./vilaw-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,  # Effective Batch Size = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_8bit",
    fp16=True,
    logging_steps=10,
    num_train_epochs=1,             # Chạy 1 epoch (~1h trên Colab T4)
    save_strategy="no",             # Tiết kiệm dung lượng đĩa Colab
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

trainer_stats = trainer.train()

## 6. Kiểm thử Suy luận (Inference Test) Trực tiếp

In [ ]:
# Chuyển sang chế độ Inference tối ưu của Unsloth
FastLanguageModel.for_inference(model)

cau_hoi_test = "Thời hiệu khởi kiện vụ án tranh chấp hợp đồng thương mại được quy định như thế nào trong Luật Thương mại Việt Nam?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": cau_hoi_test}
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=512,
    temperature=0.3,
    repetition_penalty=1.1
)

tra_loi = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
print("=== KẾT QUẢ VILAW-LLM TRẢ LỜI ===\n")
print(tra_loi)

## 7. Lưu LoRA Adapter

In [ ]:
model.save_pretrained("vilaw-sft-lora")
tokenizer.save_pretrained("vilaw-sft-lora")
print("✓ Đã lưu LoRA Adapter tại thư mục: vilaw-sft-lora (~160MB)")